# பாடம் 11 - ஏஜென்ட்-டூ-ஏஜென்ட் (A2A) நெறிமுறை


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A நெறிமுறை என்றால் என்ன?

The **Agent-to-Agent (A2A) நெறிமுறை** ஒரு திறந்த தரநிலையாகும், இது AI ஏஜென்ட்களுக்கு தொடர்பு கொள்ள, ஒருவரை ஒருவர் கண்டறிய, மற்றும் ஒத்துழைக்க உதவுகிறது — அவை வெவ்வேறு கட்டமைப்புகளில் உருவாக்கப்பட்டாலும் அல்லது வெவ்வேறு சேவைகளால் ஹோஸ்ட் செய்யப்பட்டாலும் கூட.

Key concepts:

- **Discovery** – ஏஜென்ட்கள் தங்களின் திறன்களை விளக்கும் *Agent Card* ஐ வெளியிடுகின்றன, இது மற்ற ஏஜென்ட்கள் (அல்லது ஒழுங்குபடுத்திகள்) ஒரு காரியத்திற்கான சரியான நிபுணரைக் கண்டுபிடிக்க எளிதாகிறது.
- **Message Passing** – ஏஜென்ட்கள் ஒரு பொதுவான நெறிமுறை மூலம் கட்டமைக்கப்பட்ட செய்திகள் பரிமாறிக்கொள்கிறன, ஆகையால் ஒரு ஏஜென்டின் கோரிக்கையை அதன் உள்ளக அமலாக்கத்தால் பாதிக்கப்படாமல் மற்றொரு ஏஜென்ட் புரிந்து கொண்டு நிறைவேற்ற முடியும்.
- **Task Lifecycle** – A2A *submitted*, *working*, *completed*, மற்றும் *failed* போன்ற நிலையுகளை வரையறுக்கிறது, இது ஒப்படைக்கப்பட்ட பணியின் முன்னேற்றத்தை ஒழுங்குபடுத்தியவருக்கு முழுமையான தெளிவாக காட்டுகிறது.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## நிபுணத்துவமிக்க பயண முகவர்களை உருவாக்குதல்


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## வேலைநடைமுறை மூலம் பல-ஏஜெண்ட் ஒத்துழைப்பு

A2A செய்தி பரிமாற்றத்தை பிரதிபலிக்கும் வரிசைப்படியான ஒரு வேலைநடைமுறையில் நாம் இந்த மூன்று ஏஜெண்ட்களையும் இணைக்கிறோம்:

1. **CurrencyExchangeAgent** பயனரின் கோரிக்கையை பெறுகிறது மற்றும் நாணய வழிகாட்டுதலை உருவாக்குகிறது.
2. **ActivityPlannerAgent** மேம்படுத்தப்பட்ட சூழ்நிலையை பெறுகிறது மற்றும் செயல்பாடுகளுக்கான பரிந்துரைகளைச் சேர்க்கிறது.
3. **TravelManagerAgent** இரு உள்ளீடுகளையும் ஒருங்கிணைத்து இறுதி பயண சுருக்கத்தை உருவாக்குகிறது.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## உற்பத்தி சூழலில் A2A-ஐப் புரிந்துகொள்ளுதல்

உற்பத்தி சூழலில் A2A நெறிமுறை பல்வேறு சேவைகள் மத்தியில் சக்திவாய்ந்த செயல்திறன் சூழ்நிலைகளை திறக்கிறது:

| திறன் | விளக்கம் |
|---|---|
| **வெவ்வேறு கட்டமைப்புகளுக்கிடையேயான இணக்கம்** | ஒரு கட்டமைப்பில் உருவாக்கப்பட்ட ஏஜென்ட் A2A-ஐ பின்பற்றும் வேறு எந்த கட்டமைப்பிலும் உருவாக்கப்பட்ட ஏஜென்ட்டுக்கும் பணிகளை ஒப்படைக்கலாம், இதனால் அமைப்புகளுக்கு மத்தியில் உண்மையான இணக்கத்தன்மை உருவாகிறது. |
| **சேவை எல்லைகள்** | ஏஜென்ட்கள் தனித்தனியான மைக்ரோசேவைகள், மேக பிராந்தியங்கள் அல்லது வித்தியாசமான நிறுவனங்களில் இருந்து இருந்தாலும் கூட தடையின்றி ஒத்துழைக்க முடியும். |
| **டைனமிக் கண்டறிதல்** | ஓர் ஒર્કெஸ்ட்ரேட்டர் இயக்கநேரத்தில் Agent Card registry-ஐ கேள்வி கேட்டு, குறிப்பிட்ட துணைப் பணிக்கான சிறந்த நிபுணரை கண்டுபிடிக்கலாம். |
| **ஸ்ட்ரீமிங் & புஷ் அறிவிப்புகள்** | A2A நேரடி முன்னேற்றத் தகவலுக்காக Server-Sent Events (SSE)-ஐ ஆதரிக்கிறது மற்றும் நீண்டகாலம் நடைபெறும் பணிகளுக்காக புஷ் அறிவிப்புகளை ஆதரிக்கிறது. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
பயன்பாட்டில் ஒவ்வொரு ஏஜென்டும் ஒரு HTTP endpoint-ஐ வெளிப்படுத்தி, Agent Card-ஐ வெளியிட்டு, மற்றும்
A2A JSON-RPC நெறிமுறையின் மூலம்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டவை:

1. **A2A நெறிமுறை என்பது என்ன** — முகவர்-முகவர் கண்டுபிடிப்பு, செய்தி பரிமாற்றம்,
   மற்றும் பணி மேலாண்மைக்கு ஒரு திறந்த தரநிலை.
2. **சிறப்பு முகவர்களை எப்படி உருவாக்குவது** — ஒரு நாணய பரிமாற்ற முகவர், ஒரு செயற்பாட்டு திட்டமிடும் முகவர்,
   மற்றும் ஒரு பயண மேலாளர் ஒழுங்குபடுத்தி.
3. **முகவர்களை ஒரு வேலையாக்கத்தில (workflow) இணைப்பது எப்படி** — `WorkflowBuilder` ஐப் பயன்படுத்தி தொடர்ச்சியான
   முகவர்களுக்கு இடையேயான செய்தி பரிமாற்றத்தை மாதிரியாக்க.
4. **A2A உற்பத்தி சூழலில் எப்படி செயல்படுகிறது** — டைனமிக் கண்டுபிடிப்பு மற்றும் ஸ்ட்ரீமிங் புதுப்பிப்புகளுடன் பல்வேறு ஃப்ரேம்வொர்க்குகளுக்கும் சேவைகளுக்கும் இடையிலான ஒத்துழைப்பை சாத்தியம் செய்கிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு அறிவிப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளுங்கள். தன் சொந்த மொழியில் உள்ள முதலையிருக்கும் ஆவணம் தான் அதிகாரப்பூர்வ ஆதாரம் என கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பாளரை அணுகுவதையே பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்தப் புரிதல் தவறுகளுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
